In [9]:
import pygsti
from pygsti.extras import ml
from pygsti.circuits import Circuit
from pygsti.processors.processorspec import QubitProcessorSpec as QPS
from matplotlib import pyplot as plt
import numpy as np
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
num_qubits = 4

gate_names = ['Gxpi2', 'Gypi2', 'Gcphase']

availability = {'Gcphase': [(0,1), (1,2), (2,3), (3,0)]} # This specifies what qubits a gate can act on. For example, the above says that the Gcphase gate can only act on qubits 0 and 1, or 1 and 2, etc. If a gate is not specified in this dictionary, it is assumed to be available on all qubits (e.g. Gxpi2 and Gypi2 are available on all qubits).

qubit_labels = [i for i in range(num_qubits)]

pspec = QPS(num_qubits = num_qubits, qubit_labels=qubit_labels,
            gate_names=gate_names, availability=availability)

circuit = Circuit('[Gypi2:3Gypi2:1]Gcphase:0:1[Gxpi2:0Gypi2:1Gcphase:2:3]@(0,1,2,3)')

all_gates = []
for gate in pspec.gate_names:
    all_gates += list(pspec.available_gatelabels(gate, pspec.qubit_labels))

encoder = ml.encoding.StandardCircuitEncoder(pspec)

assert(encoder.depth(0) == 0)
assert(encoder.depth(10) == 10)
assert(encoder.gate_indexing == all_gates)

correct_encoding = np.array([[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], 
                             [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
                             [1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]])

assert(np.allclose(encoder.layer_encoding(circuit.layer(0)), correct_encoding[0,:]))
assert(np.allclose(encoder.layer_encoding(circuit.layer(1)), correct_encoding[1,:])) 
assert(np.allclose(encoder.layer_encoding(circuit.layer(2)), correct_encoding[2,:]))
assert(np.allclose(encoder(circuit), correct_encoding))

circuits = [circuit,circuit]

correct_circuits_tensor = np.zeros((2, 3, len(all_gates)), float)
correct_circuits_tensor[0, :, :] = correct_encoding
correct_circuits_tensor[1, :, :] = correct_encoding

circuits_tensor = ml.encoding.circuits_to_tensor(circuits, encoder, encoding_depth=None)
assert(np.allclose(circuits_tensor, correct_circuits_tensor))

assert([1, 5, 8, 9] == encoder.indices_for_qubits((1,)))
assert([0, 4, 8, 11] == encoder.indices_for_qubits((0,)))
assert([0, 1, 4, 5, 8, 9, 11] == encoder.indices_for_qubits((1, 0)))

#nbit_strings = [ '0000', '0001', '0010', '0011', '0100', '0101', '0110', '0111', 
#                 '1000', '1001', '1010', '1011', '1100', '1101', '1110', '1111']
#
#model = pygsti.models.create_crosstalk_free_model(pspec)
#probs = model.probabilities(circuit)
#correct_probabilities = [probs[bs] for bs in nbit_strings]

# Produced using the above code
correct_probabilities = np.array([0., 0., 0., 0., 0.25, 0.25, 0., 0., 0., 0., 0., 0., 0.25, 0.25, 0., 0.])

modelled_error_generators = [('H',('ZIII',)),('S',('IIIZ',)), ('S',('IXII',))]

tensors = ml.encoding.error_generator_tensors(circuits, modelled_error_generators, pspec, alpha_representation='matrix')
indices = tensors['indices']
signs = tensors['signs']
probabilities = tensors['probabilities']
alphas = tensors['alphas']

In [3]:
assert(indices.shape == (len(circuits), circuit.depth, len(modelled_error_generators)))
assert(signs.shape == (len(circuits), circuit.depth, len(modelled_error_generators)))
assert(probabilities.shape == (len(circuits), 2 ** num_qubits))

assert(np.allclose(probabilities[0,:], correct_probabilities))

index_0 = ml.errgentools.error_generator_index('H',('ZIII',))
index_1 = ml.errgentools.error_generator_index('S',('IIIZ',))
index_2 = ml.errgentools.error_generator_index('S',('IXII',))

# This error generator is unchanged by any of the layers in the circuit
assert(np.allclose(indices[0,:,1], index_1))
assert(np.allclose(signs[0,:,1], 1))

# The Gxpi2:0 gate in the last layer transforms H_{ZIII} to -H_{YIII}
index_0_transformed = ml.errgentools.error_generator_index('H',('YIII',))
assert(np.allclose(indices[0,0,0], index_0_transformed))
assert(np.allclose(signs[0,0,0], -1))
assert(np.allclose(indices[0,1,0], index_0_transformed))
assert(np.allclose(signs[0,1,0], -1))

# The error generators for the last layer should always be unchanged
assert(np.allclose(indices[0,-1,0], index_0))
assert(np.allclose(indices[0,-1,1], index_1))
assert(np.allclose(signs[0,-1,0], 1))
assert(np.allclose(signs[0,-1,1], 1))

# S_{IXII} should have an effect b/c the marginal distribution on the 2nd qubit is
# a definite outcome (1), and this element of the `alpha` matrix should have been
# filled in as it occurs as and end-of-circuit-error-generator.
assert(alphas[0,-3,index_2] == -0.25)

In [4]:
pspec_2 = QPS(num_qubits = 2, qubit_labels=[i for i in range(2)],
            gate_names=gate_names, availability={'Gcphase': [(0,1),]})

circuit_2 = Circuit('[Gxpi2:0Gypi2:1]@(0,1)')

alphas_2 = ml.encoding.dense_alpha_matrix(circuit_2, num_qubits=2, populate_for_error_generators=None)
alphas_2_reduced = ml.encoding.dense_alpha_matrix(circuit_2, num_qubits=2, populate_for_error_generators=[4, 25, 30])

# Should be equal for all the error generators we've populated it for.
assert(np.all(alphas_2[:,4] == alphas_2_reduced[:,4]))
assert(np.all(alphas_2[:,25] == alphas_2_reduced[:,25]))
assert(np.all(alphas_2[:,30] == alphas_2_reduced[:,30]))

circuit_3 = Circuit('[Gcphase:0:1]@(0,1)')
alphas_3 = ml.encoding.dense_alpha_matrix(circuit_3, num_qubits=2, populate_for_error_generators=None)
# This is a definite-outcome circuit so has no sensitivity to Hamiltonian parameters at first-order
assert(np.allclose(0., alphas_3[:,0:4**2]))

alphas_2_reduced_added = ml.encoding.dense_alpha_matrix(circuit_2, num_qubits=2, populate_for_error_generators=[10,],
                                               existing_alpha_matrix=alphas_2_reduced)

# Should not have over-written the terms previously computed
assert(np.all(alphas_2_reduced_added[:,4] == alphas_2_reduced[:,4]))
# Should have added in the terms for the 10th error generator
assert(np.all(alphas_2_reduced_added[:,10] == alphas_2[:,10]))

In [ ]:
adjacency_matrix = ml.snippers.undirected_adjacency_matrix_from_edges(availability['Gcphase'], qubit_labels)

# All indices associated with gates within 1 step on the graph, which for qubit 0 is qubits {0, 1, 3}, and so on.
correct_snipper = [[0, 1, 3, 4, 5, 7, 8, 9, 10, 11], [0, 2, 3, 4, 6, 7, 8, 9, 10, 11], [0, 1, 2, 4, 5, 6, 8, 9, 10, 11]]
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=1)
assert(correct_snipper == snipper)

# All indices associated with gates within 0 steps on the graph, which for qubit 0 is just qubit 0, and so on.
correct_snipper = [[0, 4, 8, 11], [3, 7, 10, 11], [1, 5, 8, 9]]
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=0)
assert(correct_snipper == snipper)

In [6]:
# Testing succesfully creation of a QPANN
adjacency_matrix = ml.snippers.undirected_adjacency_matrix_from_edges(availability['Gcphase'], qubit_labels)
modelled_error_generators = [('H',('ZIII',)),('S',('IIIZ',)), ('S',('IXII',))]
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=1)
x = [circuits_tensor, signs, indices, alphas, probabilities]
qpann_1 = ml.qpanns.QPANN(encoder.length, modelled_error_generators, snipper, probability_computation='expanded')
output_1 = qpann_1(x)
weights_1 = qpann_1.weights
assert(output_1.shape == (2, 16))

E0000 00:00:1778526080.919832  570412 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [7]:
tensors_2 = ml.encoding.error_generator_tensors(circuits, modelled_error_generators, pspec, alpha_representation='concise')
indices_2 = tensors_2['indices']
signs_2 = tensors_2['signs']  # Not needed by the QPANN but still returned
probabilities_2 = tensors_2['probabilities'] # Not needed by the QPANN but still returned
alphas_2 = tensors_2['alphas']

100%|██████████| 2/2 [00:00<00:00, 38.22it/s]


In [8]:
# Testing succesfully creation of a QPANN
adjacency_matrix = ml.snippers.undirected_adjacency_matrix_from_edges(availability['Gcphase'], qubit_labels)
modelled_error_generators = [('H',('ZIII',)),('S',('IIIZ',)), ('S',('IXII',))]
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=1)
x = [circuits_tensor, alphas_2, probabilities_2]
qpann_2 = ml.qpanns.QPANN(encoder.length, modelled_error_generators, snipper, probability_computation='concise')
# Call it once to initalize the weights
_ = qpann_2(x)
# Set the weights to be the same as the other QPANN so that we should get the same output
qpann_2.set_weights(weights_1)
output_2 = qpann_2(x)
assert(output_2.shape == (2, 16))
assert(np.allclose(output_1, output_2))